In [1]:
from ingest import load_faq_data
documents = load_faq_data()

In [2]:
documents[10]

{'id': '316180784f',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'}

In [3]:
documents_llm = []

for doc in documents:
    if doc['course'] == 'llm-zoomcamp':
        documents_llm.append(doc)
        
len(documents_llm)

139

In [4]:
documents = documents_llm

In [5]:
doc = documents[0]
print(doc['id'])
print(doc['question'])
print(doc['answer'])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


##### Generating questions with structured output

In [6]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions : list[str]

In [7]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [8]:
# Call the LLM for one document to see how it performs
from openai import OpenAI

openai_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"  
)

In [9]:
# Prepare the document as JSON 
import json

user_prompt = json.dumps(doc)

In [10]:
# Create the messages:
messages = [
    {
        "role": "system",
        "content": data_gen_instructions
    },
    {
        "role": "user",
        "content": user_prompt
    }
]

In [11]:
# Call the LLM to generate questions
response = openai_client.chat.completions.parse(
    model="llama3.2",
    messages=messages,
    response_format=Questions
)

In [12]:
result = response.choices[0].message.parsed

result.questions

["Is it okay if I sign up after they've started accepting projects? Will I get a certificate?",
 "How much of the course can I still do if I join late and don't submit a project?",
 'Are late enrollments usually allowed or will I be stuck on a different section?',
 'What kind of accommodations can I expect for joining really late, like after projects have already been accepted?',
 'If I miss the certificate submission window early on, will I still get to continue learning? Are there any credits at all?',
 'Do you guys accept students who are, um, a bit behind or have other obligations before we start?',
 'Will I be able to stay in touch with anyone from previous zoomcamp cohorts?',
 '...',
 "What does 'accepting submissions' mean for people joining late? Is there any catch?"]

##### Reusable Utilities

In [13]:
from evaluation_utils import llm_structured

In [14]:
# Use it on the same document
result, usage = llm_structured(
    instructions=data_gen_instructions,
    user_prompt=user_prompt,
    output_type=Questions,
    client=openai_client
)

result.questions

["Can I still enroll in the course and get a certificate even if I'm late with my first assignment?",
 "To get a certificate, it's highly recommended that students submit their projects on time, as our grading process takes place after submission deadlines.",
 "What is the best way to catch up if I've missed one of my classes and can't receive any credits for that session?",
 "Can I still join the course once it's open or does this only apply to those who've registered prior?",
 'Is there any accommodation made for late enrollments regarding the final grade?',
 'Will the instructor be accepting any additional assignments from students who join later in the course?']

In [15]:
from evaluation_utils import calc_price

cost = calc_price(usage)

cost

{'input_cost': 0.00014025, 'output_cost': 0.000639, 'total_cost': 0.00077925}

In [16]:
# Convert questions into ground truth records
records = []

for question in result.questions:
    records.append({
        "question": question,
        "document": doc['id']
    })
    
records

[{'question': "Can I still enroll in the course and get a certificate even if I'm late with my first assignment?",
  'document': '74eb249bbf'},
 {'question': "To get a certificate, it's highly recommended that students submit their projects on time, as our grading process takes place after submission deadlines.",
  'document': '74eb249bbf'},
 {'question': "What is the best way to catch up if I've missed one of my classes and can't receive any credits for that session?",
  'document': '74eb249bbf'},
 {'question': "Can I still join the course once it's open or does this only apply to those who've registered prior?",
  'document': '74eb249bbf'},
 {'question': 'Is there any accommodation made for late enrollments regarding the final grade?',
  'document': '74eb249bbf'},
 {'question': 'Will the instructor be accepting any additional assignments from students who join later in the course?',
  'document': '74eb249bbf'}]

##### Generating Ground Truth for all Documents

In [17]:
from evaluation_utils import llm_structured_retry

In [18]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)
    
    out, usage = llm_structured_retry(
        client=openai_client,
        instructions=data_gen_instructions,
        user_prompt=user_prompt,
        output_type=Questions
    )
    
    results = []
    
    for question in out.questions:
        results.append({
            "question": question,
            "document": doc['id'],
            "course": doc['course']
        })
        
    return results, usage

In [19]:
generate_ground_truth(doc)

([{'question': "If I don't participate in live sessions, will I be able to get my certificate?",
   'document': '74eb249bbf',
   'course': 'llm-zoomcamp'},
  {'question': 'What is the deadline for submitting my project to receive the certificate?',
   'document': '74eb249bbf',
   'course': 'llm-zoomcamp'},
  {'question': "Can I still join the course if I've already purchased the class materials?",
   'document': '74eb249bbf',
   'course': 'llm-zoomcamp'},
  {'question': "How do I get started with the course once it's finished?",
   'document': '74eb249bbf',
   'course': 'llm-zoomcamp'},
  {'question': 'Will there be additional fees for the certificate',
   'document': '74eb249bbf',
   'course': 'llm-zoomcamp'}],
 CompletionUsage(completion_tokens=85, prompt_tokens=187, total_tokens=272, completion_tokens_details=None, prompt_tokens_details=None))

In [20]:
# Try for first 5 documents
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)
    
# This works but is slow because it runs sequentially.

  0%|          | 0/5 [00:00<?, ?it/s]

##### Parallel Processing

In [21]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [22]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/139 [00:00<?, ?it/s]

In [23]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)
    
len(ground_truth), len(usages)

(700, 139)

In [24]:
ground_truth[10]

{'question': "I'm having trouble accessing video/zoom links for Office Hours. Can someone please provide me with the Youtube Live link I can use instead?",
 'document': '489dd1c9d9',
 'course': 'llm-zoomcamp'}

In [25]:
# Calculate the total cost of generating the ground truth data
total_cost = 0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]
    
total_cost

0.12829874999999996

In [26]:
# with helper
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.12829874999999996

In [27]:
# Create a DataFrame so we can save the ground truth data to a CSV file
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)

In [28]:
# Save it for later use
df_ground_truth.to_csv("data/ground_truth.csv", index=False)

In [29]:
len(df_ground_truth)

700